# Deep Agents Guide — Multi-Agent Mastery Week 4

**From Demo to Production-Ready Systems**

This notebook walks you through building deep agents using the `deepagents` library — LangChain's agent harness built on LangGraph. Deep agents go beyond simple ReAct loops with built-in planning, file system tools, subagent spawning, and long-term memory.

**What you'll build:**
1. A minimal deep agent with a custom tool
2. A research agent that searches the web and writes reports
3. A multi-agent system with subagents
4. Streaming for real-time output
5. Observability with LangSmith tracing

**Docs:** [Deep Agents Overview](https://docs.langchain.com/oss/python/deepagents/overview) | [Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart) | [Customization](https://docs.langchain.com/oss/python/deepagents/customization)

---
## 0. Setup & Installation

In [ ]:
# Install the deep agents SDK, OpenAI provider, and search tool
!pip install -qU deepagents "langchain[openai]" tavily-python python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set your API keys (replace with your actual keys or use a .env file)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "your-tavily-api-key")

# Enable LangSmith tracing (Part 8)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "your-langsmith-api-key")
os.environ["LANGCHAIN_PROJECT"] = "deep-agent-demo"

---
## 1. Your First Deep Agent

The simplest possible deep agent — a single custom tool, no search, no subagents.

`create_deep_agent` gives you an agent with **built-in capabilities** out of the box:
- **Planning** via `write_todos` — breaks complex tasks into steps
- **File system** — `read_file`, `write_file`, `edit_file`, `ls` for context management
- **Subagent spawning** — delegate subtasks to isolated child agents

You just add your own tools on top.

In [ ]:
from deepagents import create_deep_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]}
)

print(result["messages"][-1].content)

**What happened under the hood:**
1. The agent received your message
2. It recognized it needed weather data and called `get_weather`
3. It synthesized the tool result into a natural language response

This looks simple, but unlike a basic chain, the agent *decided* to call that tool. If you asked a question that didn't need weather, it would skip it entirely.

---
## 2. Research Deep Agent

Now let's build something real — a research agent that searches the web and writes reports.

This is a **deep agent** because it:
- **Plans** its research approach using the built-in todo tool
- **Searches** multiple times with refined queries
- **Manages context** by writing intermediate results to its file system
- **Synthesizes** findings into a coherent report

In [ ]:
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search and return results with titles, URLs, and content snippets."""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [ ]:
research_instructions = """You are an expert researcher. Your job is to conduct 
thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering 
information.

## Workflow
1. Plan your research — break the topic into sub-questions using write_todos
2. Search for each sub-question
3. Save intermediate findings to files to manage context
4. Synthesize everything into a final report

## `internet_search`
Use this to run an internet search for a given query. You can specify the max 
number of results to return, the topic, and whether raw content should be included.
"""

research_agent = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [ ]:
result = research_agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangGraph and how does it compare to other agent frameworks?"}]}
)

print(result["messages"][-1].content)

**What makes this "deep":**
- The agent autonomously planned multiple search queries
- It used `write_file` to save raw search results, preventing context overflow
- It iterated — if first results were insufficient, it searched again with refined queries
- It synthesized all findings into a structured report

A shallow agent would do: search once → return result. This agent does: plan → search → evaluate → refine → search again → synthesize.

---
## 3. Choosing Your Model

We're using OpenAI throughout this notebook, but you can swap to any provider.
Use the `provider:model` shorthand for quick switching.

In [ ]:
# Option A: Provider shorthand — what we use in this notebook (easiest)
agent_openai = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

# Option B: Full control with init_chat_model
from langchain.chat_models import init_chat_model

model = init_chat_model(model="openai:gpt-4.1")
agent_custom = create_deep_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

# Option C: Other providers (requires their API keys)
# agent_claude = create_deep_agent(model="claude-sonnet-4-5-20250929", ...)
# agent_gemini = create_deep_agent(model="google_genai:gemini-2.5-flash-lite", ...)

---
## 4. Subagents — Multi-Agent Spawning

Deep agents can spawn **subagents** to isolate complex subtasks. This is the deep agents equivalent of the supervisor → worker pattern we covered in the slides.

**Why subagents?**
- Keep the main agent's context clean
- Specialized prompts/tools for specific subtasks
- Failure isolation — a subagent crash doesn't kill the parent

Each subagent gets its own:
- System prompt
- Tools
- Model (optional override)
- Isolated context

In [ ]:
from deepagents import create_deep_agent

# Define a research subagent with its own tools and prompt
research_subagent = {
    "name": "research-agent",
    "description": "Used to research topics in depth. Delegates to this agent when detailed web research is needed.",
    "system_prompt": """You are a thorough researcher. Search the web for information,
evaluate what you find, and return a comprehensive summary. Always cite your sources.""",
    "tools": [internet_search],
}

# Define an analysis subagent (no special tools — just LLM reasoning)
analysis_subagent = {
    "name": "analysis-agent",
    "description": "Used to analyze and synthesize information. Delegates to this agent for critical thinking tasks.",
    "system_prompt": """You are an expert analyst. Given research findings, identify
key themes, gaps, contradictions, and actionable insights. Be specific and structured.""",
    "tools": [],
}

# The parent agent orchestrates both subagents
orchestrator = create_deep_agent(
    model="openai:gpt-4.1",
    system_prompt="""You are a research coordinator. When the user asks a question:
1. Use the research-agent to gather information
2. Use the analysis-agent to analyze the findings
3. Synthesize everything into a final answer

Always use subagents for their specialized tasks rather than doing everything yourself.""",
    subagents=[research_subagent, analysis_subagent],
)

In [ ]:
result = orchestrator.invoke(
    {"messages": [{"role": "user", "content": "What are the main approaches to AI agent memory and what are the tradeoffs?"}]}
)

print(result["messages"][-1].content)

**What happened:**
1. The orchestrator received the question
2. It spawned `research-agent` to search the web for information on agent memory
3. It spawned `analysis-agent` to analyze the research findings
4. It synthesized both outputs into a final response

The subagents run in isolation — their internal tool calls and intermediate state don't pollute the parent's context.

---
## 5. Streaming — Real-Time Agent Output

In production, users shouldn't stare at a blank screen for 30 seconds. Deep agents have built-in streaming support via LangGraph.

In [ ]:
simple_agent = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[internet_search],
    system_prompt="You are a helpful research assistant. Be concise.",
)

# Stream events as they happen
for event in simple_agent.stream(
    {"messages": [{"role": "user", "content": "What is the latest news about LangGraph?"}]},
    stream_mode="values",
):
    messages = event.get("messages", [])
    if messages:
        last_msg = messages[-1]
        if hasattr(last_msg, "content") and last_msg.content:
            # Only print the latest message content (truncated if long)
            content = str(last_msg.content)
            print(f"[{last_msg.type}] {content[:300]}{'...' if len(content) > 300 else ''}")
            print()

### Streaming Modes

| Mode | What you get | When to use |
|------|-------------|-------------|
| `values` | Full state after each step | Debugging, UI state display |
| `updates` | Only changes from each node | Progress tracking, logging |
| `messages` | LLM tokens one-by-one | ChatGPT-style streaming UX |
| `custom` | Your own data via `stream_writer` | Status updates, progress bars |
| `debug` | Everything — all events | Development only |

---
## 6. Structured Output

Need the agent to return data in a specific format? Use `response_format` with a Pydantic model.

In [ ]:
from pydantic import BaseModel, Field


class CompanyBrief(BaseModel):
    """A structured company research brief."""
    company_name: str = Field(description="The name of the company")
    industry: str = Field(description="Primary industry or sector")
    summary: str = Field(description="2-3 sentence overview of the company")
    key_products: list[str] = Field(description="Main products or services")
    recent_news: str = Field(description="Most notable recent development")
    competitive_position: str = Field(description="Brief competitive analysis")


structured_agent = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[internet_search],
    system_prompt="You are a business research analyst. Research companies thoroughly.",
    response_format=CompanyBrief,
)

result = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "Research Anthropic"}]}
)

brief = result["structured_response"]
print(f"Company: {brief.company_name}")
print(f"Industry: {brief.industry}")
print(f"Summary: {brief.summary}")
print(f"Products: {brief.key_products}")
print(f"Recent News: {brief.recent_news}")
print(f"Position: {brief.competitive_position}")

---
## 7. Human-in-the-Loop

For sensitive operations, you can require human approval before the agent executes certain tools.

In [ ]:
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from deepagents import create_deep_agent


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"


@tool
def draft_email(to: str, subject: str, body: str) -> str:
    """Draft an email (does not send)."""
    return f"Draft created — To: {to}, Subject: {subject}\n\n{body}"


checkpointer = MemorySaver()

safe_agent = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[send_email, draft_email],
    system_prompt="You are an email assistant.",
    interrupt_on={
        "send_email": True,     # Requires approval before sending
        "draft_email": False,   # No approval needed for drafts
    },
    checkpointer=checkpointer,
)

print("Agent created with human-in-the-loop on send_email.")
print("When send_email is called, execution pauses for your approval.")

---
## 8. Observability — LangSmith Tracing

If you set the environment variables in cell 0, **all agent runs are already being traced** in LangSmith.

Go to [smith.langchain.com](https://smith.langchain.com) → select the `deep-agent-demo` project → inspect traces.

### What to look for in traces:

| Metric | Why it matters |
|--------|---------------|
| **Token usage per step** | Find your biggest cost driver |
| **Latency breakdown** | Find the bottleneck (usually search or verbose LLM calls) |
| **Tool call success/failure** | Are tools failing silently? |
| **Agent loops** | Is the agent stuck retrying the same thing? |
| **Subagent traces** | Drill into what each subagent did internally |

### Quick verification:

In [ ]:
# Verify tracing is active
print(f"Tracing enabled: {os.environ.get('LANGCHAIN_TRACING_V2', 'false')}")
print(f"Project: {os.environ.get('LANGCHAIN_PROJECT', 'not set')}")
print()
print("If tracing is 'true', all agent.invoke() and agent.stream() calls")
print("above have been logged to LangSmith. Check your dashboard.")
print()
print("Dashboard: https://smith.langchain.com")

### Alternatives to LangSmith

- **[Langfuse](https://langfuse.com/)** — Open-source, self-hostable
- **[Arize Phoenix](https://phoenix.arize.com/)** — Great for evaluation and experimentation
- **[Braintrust](https://www.braintrust.dev/)** — Strong eval framework

LangSmith integrates natively with LangGraph/Deep Agents, but any of these work well.

---
## 9. Deploy with FastAPI

Wrap your deep agent in a FastAPI endpoint with streaming. Save this as `server.py` and run with `uvicorn server:app`.

In [ ]:
# This cell shows the deployment code — save as server.py to run

deployment_code = '''
import os
import json
from typing import Literal
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from tavily import TavilyClient
from deepagents import create_deep_agent

# --- Setup ---
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "deep-agent-production"

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
):
    """Run a web search."""
    return tavily_client.search(query, max_results=max_results, topic=topic)


agent = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[internet_search],
    system_prompt="You are an expert research assistant. Be thorough but concise.",
)

app = FastAPI(title="Deep Agent API")


class ResearchRequest(BaseModel):
    question: str


@app.post("/research")
async def research(req: ResearchRequest):
    """Run research agent and stream results."""

    async def stream():
        try:
            async for event in agent.astream_events(
                {"messages": [{"role": "user", "content": req.question}]},
                version="v2",
            ):
                kind = event.get("event")
                if kind == "on_chat_model_stream":
                    chunk = event["data"].get("chunk")
                    if chunk and hasattr(chunk, "content") and chunk.content:
                        yield f"data: {json.dumps({\"content\": chunk.content})}\\n\\n"
            yield "data: [DONE]\\n\\n"
        except Exception as e:
            yield f"data: {json.dumps({\"error\": str(e)})}\\n\\n"

    return StreamingResponse(stream(), media_type="text/event-stream")


@app.post("/research/sync")
async def research_sync(req: ResearchRequest):
    """Run research agent and return full result."""
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": req.question}]}
        )
        return {"answer": result["messages"][-1].content}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

print(deployment_code)
print("\n" + "=" * 60)
print("Save as server.py, then run:")
print("  uvicorn server:app --reload --port 8000")
print("\nTest with:")
print('  curl -X POST http://localhost:8000/research/sync -H "Content-Type: application/json" -d \'{"question": "What is LangGraph?"}\'')

---
## 10. Production Checklist

Before shipping a deep agent to real users:

- [ ] **Timeouts** — set `recursion_limit` on the agent to prevent infinite loops
- [ ] **Error handling** — wrap agent calls in try/except with graceful fallbacks
- [ ] **Rate limiting** — protect your API keys and your budget
- [ ] **Logging / Tracing** — LangSmith or equivalent, always on in production
- [ ] **Cost controls** — track token usage per request, set budget alerts
- [ ] **Testing** — build an eval dataset, run before every deploy
- [ ] **Streaming** — users need to see progress, not wait 30s for nothing

---

## Resources

- [Deep Agents Overview](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)
- [Deep Agents Customization](https://docs.langchain.com/oss/python/deepagents/customization)
- [LangGraph Agent Architectures](https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/)
- [LangGraph Multi-Agent Concepts](https://langchain-ai.github.io/langgraph/concepts/multi_agent/)
- [LangSmith Docs](https://docs.smith.langchain.com/)
- [DeepLearning.AI — AI Agents in LangGraph](https://www.deeplearning.ai/short-courses/ai-agents-in-langgraph/)